# Wolfson — Training Notebook

Trains the jazz sax phrase model on the Weimar Jazz Database.

**Steps:**
1. Clone the Wolfson repo
2. Mount Google Drive and upload `wjazzd.db`
3. Install dependencies
4. Prepare training data
5. Train the model
6. Save the trained model to Google Drive

**Runtime:** Set to GPU (Runtime → Change runtime type → T4 GPU) before starting.

## 1. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/davidderoure/wolfson.git
%cd wolfson
!pip install -q pretty_midi numpy torch

## 2. Mount Google Drive

Your Drive is used to:
- Read `wjazzd.db` (place it at `MyDrive/wolfson/wjazzd.db` before running)
- Save the trained model after training

If you prefer to upload the database directly to the Colab session instead, skip this cell and use the upload cell below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

DRIVE_WOLFSON = '/content/drive/MyDrive/wolfson'
os.makedirs(DRIVE_WOLFSON, exist_ok=True)
os.makedirs('data/raw', exist_ok=True)

db_src = f'{DRIVE_WOLFSON}/wjazzd.db'
db_dst = 'data/raw/wjazzd.db'

if os.path.exists(db_src):
    shutil.copy(db_src, db_dst)
    print(f'Copied wjazzd.db from Drive ({os.path.getsize(db_dst)/1e6:.1f} MB)')
else:
    print(f'wjazzd.db not found at {db_src}')
    print('Either place it there, or use the direct upload cell below.')

## 2b. (Alternative) Upload wjazzd.db directly

Skip this if you used Google Drive above.

In [ ]:
# from google.colab import files
# import os
# os.makedirs('data/raw', exist_ok=True)
# uploaded = files.upload()   # select wjazzd.db
# for fname in uploaded:
#     with open(f'data/raw/{fname}', 'wb') as f:
#         f.write(uploaded[fname])
#     print(f'Saved {fname} to data/raw/')

## 3. Inspect the database

In [ ]:
!python data/prepare.py --inspect

## 4. Prepare training data

In [ ]:
!python data/prepare.py --instrument sax

## 5. Check GPU and train

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

In [ ]:
# Adjust epochs as needed. 100 epochs takes ~15-20 min on a T4.
INSTRUMENT = 'sax'
EPOCHS     = 100
BATCH_SIZE = 64

!python generator/train.py \
    --instrument {INSTRUMENT} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE}

## 6. Save trained model to Google Drive

In [ ]:
import shutil, os

INSTRUMENT = 'sax'
src  = f'models/{INSTRUMENT}_best.pt'

# Save to Drive
drive_dst = f'{DRIVE_WOLFSON}/{INSTRUMENT}_best.pt'
if os.path.exists(src):
    shutil.copy(src, drive_dst)
    print(f'Saved to Drive: {drive_dst}')
else:
    print(f'Model not found at {src} — check training completed successfully.')

In [ ]:
# Or download directly to your machine
# from google.colab import files
# files.download(f'models/{INSTRUMENT}_best.pt')

## 7. (Optional) Resume training from a saved checkpoint

In [ ]:
# INSTRUMENT = 'sax'
# checkpoint = f'{DRIVE_WOLFSON}/{INSTRUMENT}_best.pt'
#
# # Copy checkpoint into session
# import shutil, os
# os.makedirs('models', exist_ok=True)
# shutil.copy(checkpoint, f'models/{INSTRUMENT}_best.pt')
#
# !python generator/train.py \
#     --instrument {INSTRUMENT} \
#     --epochs 50 \
#     --batch-size 64 \
#     --resume models/{INSTRUMENT}_best.pt

## 8. (Optional) Train trumpet model

The WJD contains 102 trumpet solos + 15 cornet solos (117 total).

In [ ]:
# !python data/prepare.py --instrument trumpet
# !python generator/train.py --instrument trumpet --epochs 100 --batch-size 64